# Building Production Ready AI Agents with LangChain and Azure AI

## Lab Overview

In this lab you will learn how to build **AI agents using LangChain and Azure deployed models**.

Agents allow LLMs to:
- Reason about tasks
- Decide which tools to use
- Execute actions
- Return structured results

This notebook demonstrates how to integrate **Azure OpenAI deployments with LangChain**, build tools, create an agent, add memory, and implement simple retrieval.

## What You Will Learn

By the end of this lab you will be able to:

1. Install and configure LangChain for Azure
2. Connect LangChain to Azure deployed models
3. Create prompt chains
4. Build tools for agents
5. Implement reasoning agents
6. Add conversational memory
7. Implement document retrieval
8. Build a developer research assistant agent


## Architecture Overview

User  
↓  
LangChain Agent  
↓  
Azure AI Model  
↓  
Tools  
- Python functions  
- Knowledge tools  
- Retrieval tools  

This mirrors modern **enterprise AI assistant architectures**.


## Step 1 - Install Dependencies

In [1]:
!pip install -U langchain
!pip install -U langchain-openai
!pip install -U langchain-azure-ai
!pip install -U openai
!pip install -U faiss-cpu
!pip install -U python-dotenv
!pip install -U tiktoken
!pip install -U langchain_classic
!pip install langchain_community


  Attempting uninstall: langgraph

    Found existing installation: langgraph 1.0.10

    Uninstalling langgraph-1.0.10:

      Successfully uninstalled langgraph-1.0.10

   ---------------------------------------- 0/2 [langgraph]
   ---------------------------------------- 0/2 [langgraph]
   ---------------------------------------- 0/2 [langgraph]
   ---------------------------------------- 0/2 [langgraph]
   ---------------------------------------- 0/2 [langgraph]
   ---------------------------------------- 0/2 [langgraph]
   ---------------------------------------- 0/2 [langgraph]
  Attempting uninstall: langchain
   ---------------------------------------- 0/2 [langgraph]
    Found existing installation: langchain 1.2.10
   ---------------------------------------- 0/2 [langgraph]
    Uninstalling langchain-1.2.10:
   ---------------------------------------- 0/2 [langgraph]
      Successfully uninstalled langchain-1.2.10
   ---------------------------------------- 0/2 [langgraph]
 

## Step 2 - Configure Azure Credentials

In [2]:
import os

os.environ["AZURE_OPENAI_ENDPOINT"] = "https://<TBD>.cognitiveservices.azure.com/"
os.environ["AZURE_OPENAI_API_KEY"] = ""
os.environ["AZURE_OPENAI_DEPLOYMENT"] = "gpt-4o"
os.environ["OPENAI_API_VERSION"] = "2024-12-01-preview"

print("Environment variables configured.")

Environment variables configured.


## Step 3 - Connect LangChain to Azure Model

In [3]:
from langchain_openai import AzureChatOpenAI
import os

llm = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    deployment_name=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    api_version=os.environ["OPENAI_API_VERSION"],
    temperature=0.2
)

response = llm.invoke("Explain what an AI agent is.")
print(response.content)

An **AI agent** is a computational entity or system that is designed to perceive its environment, reason about it, and take actions to achieve specific goals. It operates autonomously, meaning it can make decisions and act without constant human intervention. AI agents are a fundamental concept in artificial intelligence and are used in various applications, from simple rule-based systems to complex machine learning models.

### Key Characteristics of an AI Agent:
1. **Perception**: An AI agent gathers information about its environment through sensors or data inputs. For example, a chatbot perceives user input as text, while a self-driving car uses cameras and sensors to perceive its surroundings.

2. **Reasoning**: The agent processes the information it perceives, analyzes it, and makes decisions based on predefined rules, learned patterns, or algorithms.

3. **Action**: After reasoning, the agent performs actions to influence the environment or achieve its goals. For example, a virtu

## Step 4 - Prompt Engineering with LangChain

In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an expert AI instructor.

Explain the following concept for software engineers.

Concept: {topic}
""")

chain = prompt | llm

result = chain.invoke({"topic": "LangChain agents"})
print(result.content)

### LangChain Agents Explained for Software Engineers

LangChain is a powerful framework designed to simplify the development of applications that leverage **large language models (LLMs)**, such as OpenAI's GPT or other similar models. One of its key features is **agents**, which are dynamic components that enable LLMs to interact with external tools, APIs, or data sources, and make decisions based on the information they gather.

Let’s break down the concept of **LangChain agents** step by step:

---

### **What Are LangChain Agents?**
LangChain agents are **decision-making entities** powered by LLMs that can perform tasks autonomously by interacting with external tools or systems. Unlike static prompts, agents are designed to handle complex workflows where the LLM needs to:
1. **Reason** about what actions to take.
2. **Call external tools** (e.g., APIs, databases, or Python functions).
3. **Interpret results** from those tools.
4. **Iterate** until the task is complete.

In essence,

## Step 5 - Creating Tools for the Agent

In [5]:
from langchain.tools import tool

@tool
def multiply_numbers(a:int,b:int)->int:
    """Multiply two numbers"""
    return a*b

In [6]:
from datetime import datetime
from langchain.tools import tool

@tool
def get_current_year():
    """Returns the current year"""
    return datetime.now().year

In [7]:
@tool
def explain_ai_concept(topic:str):
    """Explain an AI concept"""
    
    prompt=f"Explain {topic} for developers"
    
    return llm.invoke(prompt).content

## Step 6 - Combine Tools

In [8]:
tools=[
    multiply_numbers,
    get_current_year,
    explain_ai_concept
]

tools

[StructuredTool(name='multiply_numbers', description='Multiply two numbers', args_schema=<class 'langchain_core.utils.pydantic.multiply_numbers'>, func=<function multiply_numbers at 0x000001FFD10F7E20>),
 StructuredTool(name='get_current_year', description='Returns the current year', args_schema=<class 'langchain_core.utils.pydantic.get_current_year'>, func=<function get_current_year at 0x000001FFD10F6DE0>),
 StructuredTool(name='explain_ai_concept', description='Explain an AI concept', args_schema=<class 'langchain_core.utils.pydantic.explain_ai_concept'>, func=<function explain_ai_concept at 0x000001FFD10CF6A0>)]

## Step 7 - Create a LangChain Agent

In [9]:
from langchain_classic.agents import create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI agent. Use tools when needed."),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)



## Step 8 - Run the Agent

In [10]:
agent_executor.invoke({"input": "Multiply the current year by 2."})



> Entering new AgentExecutor chain...

Invoking: `get_current_year` with `{}`


2026
Invoking: `multiply_numbers` with `{'a': 2023, 'b': 2}`


4046The current year multiplied by 2 is **4046**.

> Finished chain.


{'input': 'Multiply the current year by 2.',
 'output': 'The current year multiplied by 2 is **4046**.'}

## Step 9 - Add Memory

In [11]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI agent. Use tools when needed."),
    MessagesPlaceholder("chat_history"),        
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),     
])

agent = create_openai_tools_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [12]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Wrap your agent executor with message history
history_store = {}

def get_session_history(session_id: str):
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]

In [13]:
agent_with_memory = RunnableWithMessageHistory(
    agent_executor,                 # your AgentExecutor
    get_session_history,
    input_messages_key="input",     # key used in invoke({"input": ...})
    history_messages_key="chat_history"
)

# Use a fixed session id for the conversation
config = {"configurable": {"session_id": "demo-session"}}

### Test Memory

In [14]:
agent_with_memory.invoke({"input": "My name is Alex"}, config=config)
agent_with_memory.invoke({"input": "What is my name?"}, config=config)



> Entering new AgentExecutor chain...
Hello, Alex! How can I assist you today?

> Finished chain.


> Entering new AgentExecutor chain...
Your name is Alex.

> Finished chain.


{'input': 'What is my name?',
 'chat_history': [HumanMessage(content='My name is Alex', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Hello, Alex! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'output': 'Your name is Alex.'}

## Step 10 - Create Documents for Retrieval

In [15]:
documents=[
"LangChain is a framework for building applications with LLMs",
"Azure OpenAI provides enterprise grade AI models",
"Agents allow LLMs to interact with external tools"
]

documents

['LangChain is a framework for building applications with LLMs',
 'Azure OpenAI provides enterprise grade AI models',
 'Agents allow LLMs to interact with external tools']

## Step 11 - Create Embeddings

In [16]:
from langchain_openai import AzureOpenAIEmbeddings

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    deployment="text-embedding-ada-002"
)

## Step 12 - Create Vector Store

In [17]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_texts(
    documents,
    embedding=embeddings
)

vector_store

## Step 13 - Create Retriever

In [18]:
retriever = vector_store.as_retriever()

retriever.invoke("What is LangChain?")

[Document(id='c77e3f49-8ad8-4671-ac7e-65ab400c3dac', metadata={}, page_content='LangChain is a framework for building applications with LLMs'),
 Document(id='a03bfff5-99fe-462e-82f5-218c23b00d09', metadata={}, page_content='Agents allow LLMs to interact with external tools'),
 Document(id='7a8d8ad6-dc6c-46fb-9b50-f95d5b4f06e8', metadata={}, page_content='Azure OpenAI provides enterprise grade AI models')]

## Summary

In this lab you learned how to:

- Connect LangChain with Azure AI deployments
- Create structured prompts
- Build tools for agents
- Implement reasoning agents
- Add conversation memory
- Implement vector search retrieval

These concepts form the foundation for:

- AI copilots
- enterprise assistants
- autonomous agents
- RAG systems
